# Modélisation baseline

## Informations générales

| Élément | Détail |
|---|---|
| **Propriétaire** | Tiéba Bamba |
| **Projet** | Talk to my data |
| **Domaine** | Banque de détail - Recouvrement & Risque |
| **Étape** | 2/3 - Modélisation baseline |
| **Notebook** | Construction et évaluation d'un modèle de référence |
| **Date de création** | 25 août 2026 |
| **Statut** | Terminé |

## Contexte métier

La direction **« Recouvrement & Risque »** d'une banque de détail souhaite renforcer sa politique de relance et de recouvrement au prochain trimestre. L'étape précédente (EDA) a permis de comprendre la structure du dataset, d'identifier la variable cible `default_payment_next_month`, son déséquilibre (~21 % de défauts), et les variables les plus liées au risque (notamment les statuts de remboursement `pay_*`).

Cette deuxième étape vise à construire un premier modèle de classification simple, servant de **référence (baseline)**, puis à présélectionner plusieurs familles de modèles candidats pour préparer l'optimisation approfondie du notebook 03.

## Objectifs de ce notebook

Ce notebook constitue la deuxième étape du projet. Il a pour objectifs de :

- charger les données préparées à partir des constats de l'étape d'EDA ;
- mettre en place le prétraitement nécessaire (encodage, gestion des modalités regroupées, feature engineering) ;
- entraîner un modèle de classification simple et interprétable ;
- évaluer ses performances avec des métriques adaptées au déséquilibre des classes (PR-AUC prioritaire, F1, Recall, Precision, matrice de confusion, `recall@topK`) ;
- présélectionner, par validation croisée sur le train uniquement, les modèles candidats les plus prometteurs pour l'optimisation à venir ;
- documenter les limites de cette première approche.

## Déroulé annoncé

1. **Initialisation** : imports et chargement des données préparées (train/test).
2. **Prétraitement** : encodage des variables catégorielles, mise en forme des features.
3. **Entraînement du modèle baseline** : choix et entraînement d'un modèle simple.
4. **Évaluation** : métriques adaptées au déséquilibre des classes, matrice de confusion, `recall@topK`.
5. **Présélection de modèles candidats** : validation croisée sur le train (6 familles de modèles), sélection de 4 finalistes selon la PR-AUC (métrique prioritaire) et des métriques secondaires.
6. **Synthèse** : constats, limites et pistes pour l'optimisation en notebook 03.

## Résultat attendu

À la fin de ce notebook, nous disposerons d'un modèle de référence évalué et documenté, ainsi que d'une short-list de 4 modèles candidats (Gradient Boosting, Random Forest, LightGBM, Logistic Regression) présélectionnés sur le train, prêts à être optimisés (hyperparamètres) puis évalués une seule fois sur le test dans le notebook 03.

# Préparation des données

la préparation des données a été faite dans data_prep.py en reprenant tout les regroupement que nous avons fait en EDA.

celle ci s'est faite en différentes etapes :

### 1. Configuration (`src/config.py`)

Avant de préparer les données, on centralise dans `config.py` toutes les constantes décidées lors de l'EDA (chemins, colonnes, regroupements de modalités, paramètres de split), pour que `data_prep.py`, `train.py` et `infer.py` s'y réfèrent au lieu de dupliquer ces choix.

**Chemins et colonnes cible/exclues :**

```python
from pathlib import Path

# Chemins
PROJECT_ROOT = Path(__file__).resolve().parent.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "credit_card_default.csv"

# Colonnes
TARGET_COLUMN = "default_payment_next_month"
ID_COLUMN = "id"
# Colonne suspecte reperee en EDA (prediction deja presente dans les donnees brutes,
# risque de fuite de donnees) : a exclure des features.
LEAKAGE_COLUMN = "predicted_default_payment_next_month"
EXCLUDED_COLUMNS = [ID_COLUMN, LEAKAGE_COLUMN]

# Familles de colonnes brutes, utilisees par le feature engineering.
# Convention chronologique corrigee (verifiee contre la doc source UCI) : le
# suffixe 1 correspond au mois le plus RECENT, le suffixe 6 au plus ANCIEN,
# pour les 3 familles. La colonne brute "pay_0" est renommee en "pay_1" des
# le chargement pour uniformiser la numerotation (cf. notebook 01, corrige).
BILL_AMT_COLUMNS = [f"bill_amt_{i}" for i in range(1, 7)]
PAY_AMT_COLUMNS = [f"pay_amt_{i}" for i in range(1, 7)]
PAY_STATUS_COLUMNS = ["pay_1", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6"]
```

`id` n'est pas une feature, et `predicted_default_payment_next_month` a été identifiée en EDA comme une colonne à risque de fuite de données (prédiction déjà présente dans les données brutes, réservée aux contrôles qualité selon les consignes du projet) : les deux sont exclues.

**Colonnes catégorielles/numériques, regroupements et encodage ordinal :**

```python
# education_level n'est plus one-hot mais ordinal + flag (cf. encode_education_level) :
# sortie de CATEGORICAL_COLUMNS, ses colonnes derivees sont dans NUMERIC_COLUMNS.
CATEGORICAL_COLUMNS = ["sex", "marital_status"]
TYPE_CAST_COLUMNS = ["sex", "marital_status"]

# NUMERIC_COLUMNS reflete les features APRES feature engineering (engineer_features
# et encode_education_level dans data_prep.py), pas les colonnes brutes du CSV :
# - bill_amt_1..6 remplacees par bill_amt_moyen/bill_amt_tendance (multicolinearite
#   0.77-0.95 mesuree en EDA)
# - pay_amt_1..6 conservees + reste_du_moyen et taux_utilisation_moyen
# - pay_1..pay_6 conservees + retard_max/nb_mois_en_retard (moins redondantes entre
#   elles, individuellement plus discriminantes que bill_amt_*)
NUMERIC_COLUMNS = [
    "limit_balance", "age",
    "pay_1", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6",
    "retard_max", "nb_mois_en_retard",
    "pay_amt_1", "pay_amt_2", "pay_amt_3", "pay_amt_4", "pay_amt_5", "pay_amt_6",
    "bill_amt_moyen", "bill_amt_tendance",
    "reste_du_moyen", "taux_utilisation_moyen",
    "education_level_ordinal", "is_education_undocumented",
]

# Regroupement des modalites decide en EDA (codes non documentes / rares)
EDUCATION_LEVEL_MAPPING = {4: 0, 5: 0, 6: 0}
MARITAL_STATUS_MAPPING = {"0": "3"}

# Encodage ordinal d'education_level (apres regroupement ci-dessus, valeurs {0,1,2,3}) :
# ordre croissant du taux de defaut observe en EDA (8% non documente, 18% etudes sup,
# 24% universite, 24% lycee). La modalite 0 n'a pas de position logique sur cette
# echelle : isolee via is_education_undocumented plutot que forcee dans l'ordre.
EDUCATION_LEVEL_ORDER = {1: 0, 2: 1, 3: 2}
```

Ces listes définissent explicitement quelles colonnes seront one-hot encodées vs passées telles quelles après feature engineering, et les mappings reprennent les décisions prises en EDA.

**Paramètres de split :**

```python
TEST_SIZE = 0.2
RANDOM_STATE = 42
```

Les mêmes valeurs que dans le notebook d'EDA, pour garder un split reproductible et cohérent entre les étapes du projet.

### 2. Préparation des données (`src/data_prep.py`)

Le fichier `data_prep.py` contient 5 fonctions qui s'appuient sur `config.py`.

**`load_and_prepare_data()`** — charge le CSV brut, renomme `pay_0` en `pay_1`, type les colonnes catégorielles et split train/test :

```python
def load_and_prepare_data():
    """Charge le CSV brut, type les colonnes categorielles et split train/test."""
    df = pd.read_csv(RAW_DATA_PATH)
    # pay_0 -> pay_1 : uniformise la numerotation avec bill_amt_*/pay_amt_* (suffixe 1
    # = mois le plus recent pour les 3 familles, cf. commentaire dans config.py).
    df = df.rename(columns={"pay_0": "pay_1"})
    df[TYPE_CAST_COLUMNS] = df[TYPE_CAST_COLUMNS].astype("str")

    df_train, df_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COLUMN],
    )
    return df_train, df_test
```

C'est l'équivalent des étapes faites manuellement dans le notebook d'EDA (lecture, renommage `pay_0`→`pay_1`, typage `sex`/`marital_status`, split stratifié), mais centralisé pour être réutilisé partout.

**`clean_categoricals(X)`** — applique le regroupement des modalités décidé en EDA :

```python
def clean_categoricals(X):
    """Regroupe les modalites non documentees/rares (regle fixe decidee en EDA).

    Cast egalement sex/marital_status en str : rend cette etape auto-suffisante
    pour que la pipeline reste correcte meme appliquee a des donnees brutes
    (infer.py) qui n'auraient pas transite par load_and_prepare_data().
    """
    X = X.copy()
    X["sex"] = X["sex"].astype(str)
    X["marital_status"] = X["marital_status"].astype(str).replace(MARITAL_STATUS_MAPPING)
    X["education_level"] = X["education_level"].replace(EDUCATION_LEVEL_MAPPING)
    return X
```

Cette fonction ne dépend d'aucune statistique calculée sur les données (règle fixe, pas de risque de fuite train/test) : elle peut donc être appliquée telle quelle sur train, test, ou de nouvelles données à l'inférence.

**`engineer_features(X)`** — ajoute les features dérivées décidées pour la modélisation avancée :

```python
def engineer_features(X):
    """Ajoute les features derivees decidees pour la modelisation avancee.

    - reste_du_moyen : ecart moyen bill_amt - pay_amt sur les 6 mois (montant
      non couvert par les paiements). bill_amt_i et pay_amt_i ne sont pas
      parfaitement alignes dans le temps, mais bill_amt_i et bill_amt_(i-1)
      sont tres correles (0.77-0.95 en EDA), donc l'approximation reste
      raisonnable.
    - taux_utilisation_moyen : bill_amt_i / limit_balance, moyenne sur 6 mois.
      limit_balance >= 10000 sur ce dataset, pas de risque de division par zero.
    - bill_amt_moyen / bill_amt_tendance : remplacent les 6 colonnes brutes
      bill_amt_1..6, tres colineaires entre elles. bill_amt_1 = mois le plus
      recent, bill_amt_6 = le plus ancien : tendance = bill_amt_1 - bill_amt_6,
      positive si le solde a augmente vers le mois le plus recent.
    - retard_max / nb_mois_en_retard : agregats de severite/persistance du
      retard, en complement (pas en remplacement) des pay_1..pay_6 bruts,
      individuellement les plus discriminants et moins redondants entre eux.
    """
    X = X.copy()

    reste_du = X[BILL_AMT_COLUMNS].to_numpy() - X[PAY_AMT_COLUMNS].to_numpy()
    X["reste_du_moyen"] = reste_du.mean(axis=1)

    X["taux_utilisation_moyen"] = X[BILL_AMT_COLUMNS].div(X["limit_balance"], axis=0).mean(axis=1)

    X["bill_amt_moyen"] = X[BILL_AMT_COLUMNS].mean(axis=1)
    X["bill_amt_tendance"] = X["bill_amt_1"] - X["bill_amt_6"]
    X = X.drop(columns=BILL_AMT_COLUMNS)

    X["retard_max"] = X[PAY_STATUS_COLUMNS].max(axis=1)
    X["nb_mois_en_retard"] = (X[PAY_STATUS_COLUMNS] > 0).sum(axis=1)

    return X
```

Justification du traitement différencié entre les deux familles : `bill_amt_*` est très redondante (corrélation moyenne 0,88 entre elles) et quasi nulle avec la cible, donc remplacée par des agrégats ; `pay_*` a une redondance plus modérée (moyenne 0,64) et un vrai effet de récence sur la cible (`pay_1`, le mois le plus récent, corrélation 0,38 ; `pay_6`, le plus ancien, 0,25), donc conservée telle quelle en plus des agrégats.

**`encode_education_level(X)`** — encode `education_level` en ordinal + flag plutôt qu'en one-hot :

```python
def encode_education_level(X):
    """Encode education_level en ordinal + flag plutot qu'en one-hot.

    Les modalites 1/2/3 (etudes sup/universite/lycee) suivent un ordre de
    risque croissant confirme en EDA (8%/18%/24%/24% de taux de defaut) : un
    encodage ordinal leur donne une seule variable exploitable directement par
    les modeles lineaires, sans perte pour les arbres.

    La modalite 0 (non documentee) n'a pas de position logique sur cette
    echelle : plutot que de la forcer arbitrairement dans l'ordre, elle est
    isolee dans un flag binaire is_education_undocumented, et la colonne
    ordinale recoit une valeur neutre (mediane des categories documentees)
    pour ces lignes.
    """
    X = X.copy()
    X["is_education_undocumented"] = (X["education_level"] == 0).astype(int)
    X["education_level_ordinal"] = X["education_level"].map(EDUCATION_LEVEL_ORDER).fillna(1)
    X = X.drop(columns=["education_level"])
    return X
```

**`build_pipeline(model)`** — assemble le nettoyage, le feature engineering, l'encodage, le scaling et le modèle en une seule pipeline sklearn :

```python
def build_pipeline(model):
    """Construit la pipeline complete : nettoyage, feature engineering, encodage, modele."""
    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
            ("numeric", StandardScaler(), NUMERIC_COLUMNS),
        ]
    )

    return Pipeline(
        steps=[
            ("clean_categoricals", FunctionTransformer(clean_categoricals)),
            ("engineer_features", FunctionTransformer(engineer_features)),
            ("encode_education_level", FunctionTransformer(encode_education_level)),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
```

Paramétrée par le modèle, cette fonction sera réutilisée telle quelle dans le notebook 03 (modélisation avancée) avec un autre estimateur. Le `ColumnTransformer` ne sélectionne que `CATEGORICAL_COLUMNS`/`NUMERIC_COLUMNS`, donc `id` et `predicted_default_payment_next_month` sont automatiquement exclus des features sans avoir besoin de les dropper explicitement.

Les colonnes numériques passent par un `StandardScaler()` (moyenne 0, écart-type 1) plutôt qu'un simple passage direct : sans ça, `LogisticRegression` ne convergeait pas correctement (`ConvergenceWarning`), les variables comme `limit_balance` (jusqu'à 800 000) écrasant numériquement des variables comme `pay_1` (entre -2 et 8) pendant l'optimisation. Le scaling n'a aucun effet négatif pour des modèles à base d'arbres qu'on pourrait tester en notebook 03.

### 3. Modele de base

Pour l'implémentation du modele de base nous lons utilser un modele linéaire simple qui est la regression logistique. ( ``LogisticRegression`` de ``scikit-learn`` )

In [1]:
# Imports des Variables et fonctions etablies dans les fichiers src/config.py et src/data_prep.py
import sys
sys.path.append("..")

from src.config import TARGET_COLUMN
from src.data_prep import load_and_prepare_data, build_pipeline

In [2]:
# modele logistique
from sklearn.linear_model import LogisticRegression

model_baseline = LogisticRegression(random_state=42 , max_iter=1000, class_weight="balanced")


##### **Justification du traitement du déséquilibre : `class_weight` plutôt que du ré-échantillonnage**

Le dataset est déséquilibré (~21 % de défauts). Deux approches usuelles pour ce problème : pondérer les classes (`class_weight`) ou ré-échantillonner les données (sous-échantillonnage de la classe majoritaire, ou sur-échantillonnage type SMOTE). On retient `class_weight="balanced"` pour ce projet, pour plusieurs raisons :

- **Aucune perte ni distorsion des données réelles.** Le sous-échantillonnage jetterait une partie des clients sans défaut (moins de données disponibles) ; le sur-échantillonnage type SMOTE génère des clients synthétiques par interpolation entre voisins — risqué ici vu nos features engineered (`reste_du_moyen`, `taux_utilisation_moyen`, `bill_amt_tendance`...), où une interpolation linéaire naïve entre deux clients réels pourrait produire des combinaisons incohérentes (ex. un `taux_utilisation_moyen` élevé avec un `reste_du_moyen` négatif improbable).
- **Aucun risque de fuite de données.** `class_weight` agit uniquement sur la fonction de perte pendant l'entraînement, sans toucher aux données elles-mêmes — pas besoin de veiller à ce que le ré-échantillonnage reste cantonné au train (SMOTE appliqué par erreur avant le split, ou dans les mauvais folds de la validation croisée, est une source classique de fuite de données).
- **Disponible nativement** dans la plupart des modèles testés (`LogisticRegression`, arbres, `RandomForestClassifier`, `LGBMClassifier`, `SVC`), donc applicable de façon homogène à travers toute la présélection de modèles de la section 5, sans complexifier la pipeline (le ré-échantillonnage nécessiterait une `Pipeline` de la librairie `imbalanced-learn`, incompatible avec la `Pipeline` sklearn standard utilisée ici).

**Limite assumée** : `GradientBoostingClassifier` de sklearn ne supporte pas nativement `class_weight` (contrairement aux autres modèles testés) — il a donc été entraîné sans pondération explicite en section 5. C'est un point à surveiller si ce modèle est retenu pour la suite (notebook 03), où l'on pourra passer par `sample_weight` au moment du `.fit()`.

In [3]:
import pandas as pd

df_train, df_test = load_and_prepare_data()

X_train, y_train = df_train.drop(columns=[TARGET_COLUMN]), df_train[TARGET_COLUMN]
# X_test/y_test : definis ici pour verifier la taille du split, mais non utilises
# dans le reste de ce notebook (reserves a l'evaluation finale du notebook 03).
X_test, y_test = df_test.drop(columns=[TARGET_COLUMN]), df_test[TARGET_COLUMN]

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")

X_train : (2372, 25) | X_test : (593, 25)


In [4]:
from sklearn import set_config
set_config(display="diagram")

pipeline_baseline = build_pipeline(model_baseline)
pipeline_baseline.fit(X_train, y_train)
pipeline_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clean_categoricals', ...), ('engineer_features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](25,)","['id','limit_balance','sex',...,'pay_amt_5','pay_amt_6', 'predicted_default_payment_next_month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,25
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function cle...00229D02447D0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


### 4. Évaluation

L'accuracy seule n'est pas adaptée ici : vu le déséquilibre des classes (~21 % de défauts), un modèle qui prédit toujours "pas de défaut" aurait déjà ~79 % d'accuracy sans aucune valeur métier. On utilise donc des métriques plus pertinentes pour ce contexte de risque :

- **Recall** : proportion des vrais défauts effectivement détectés — c'est le plus important ici, puisque manquer un client à risque coûte cher.
- **Precision** : proportion des alertes de défaut qui sont correctes.
- **F1-score** : compromis entre les deux.
- **PR-AUC** : performance globale du score de probabilité, plus informative que la ROC-AUC pour des classes déséquilibrées.
- **Matrice de confusion** : pour visualiser concrètement les erreurs (faux positifs vs faux négatifs).

Ces métriques sont calculées par **validation croisée stratifiée (5 folds) sur `X_train`/`y_train` uniquement**, avec les prédictions hors-échantillon agrégées (`cross_val_predict`) — la même méthodologie que la section 5. `X_test` n'est volontairement touché à aucun moment de ce notebook : il est réservé à l'évaluation finale unique du notebook 03, une fois le modèle définitif retenu et optimisé.

In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv_baseline = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_cv = cross_val_predict(pipeline_baseline, X_train, y_train, cv=cv_baseline, method="predict_proba")[:, 1]
y_pred_cv = (y_proba_cv >= 0.5).astype(int)

print(f"Accuracy  : {accuracy_score(y_train, y_pred_cv):.3f}")
print(f"Precision : {precision_score(y_train, y_pred_cv):.3f}")
print(f"Recall    : {recall_score(y_train, y_pred_cv):.3f}")
print(f"F1-score  : {f1_score(y_train, y_pred_cv):.3f}")
print(f"PR-AUC    : {average_precision_score(y_train, y_proba_cv):.3f}  (vs {y_train.mean():.3f} pour un modele aleatoire)")

Accuracy  : 0.778
Precision : 0.486
Recall    : 0.624
F1-score  : 0.547
PR-AUC    : 0.550  (vs 0.214 pour un modele aleatoire)


In [6]:
y_pred_train = pipeline_baseline.predict(X_train)
y_proba_train = pipeline_baseline.predict_proba(X_train)[:, 1]

print("--- Performance en resubstitution (train, fit puis predict sur lui-meme) ---")
print(f"Accuracy  : {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Precision : {precision_score(y_train, y_pred_train):.3f}")
print(f"Recall    : {recall_score(y_train, y_pred_train):.3f}")
print(f"F1-score  : {f1_score(y_train, y_pred_train):.3f}")
print(f"PR-AUC    : {average_precision_score(y_train, y_proba_train):.3f}")

--- Performance en resubstitution (train, fit puis predict sur lui-meme) ---
Accuracy  : 0.780
Precision : 0.490
Recall    : 0.646
F1-score  : 0.557
PR-AUC    : 0.572


**Résubstitution (train) vs validation croisée (out-of-fold)** :

| Métrique | Résubstitution (train) | CV (out-of-fold) |
|---|---|---|
| Accuracy | 0,780 | 0,778 |
| Precision | 0,490 | 0,486 |
| Recall | 0,646 | 0,624 |
| F1-score | 0,557 | 0,547 |
| PR-AUC | 0,572 | 0,550 |

L'écart entre la résubstitution (le modèle prédit sur les données qui ont servi à l'entraîner) et la validation croisée (chaque client prédit par un modèle qui ne l'a jamais vu) reste modéré — quelques points sur chaque métrique. C'est cohérent avec un modèle linéaire simple, peu capable de sur-apprendre le bruit du train, et confirme que la version validation croisée (colonne de droite) donne une estimation raisonnablement honnête de la performance de généralisation, sans avoir eu besoin de toucher `X_test`.

In [7]:
import plotly.graph_objects as go
from IPython.display import HTML, display

cm = confusion_matrix(y_train, y_pred_cv)

fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=["Pas de défaut (prédit)", "Défaut (prédit)"],
        y=["Pas de défaut (réel)", "Défaut (réel)"],
        colorscale="Blues",
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 16},
        colorbar={"title": "Effectif"},
    )
)

fig.update_layout(
    title="Matrice de confusion - Modèle baseline (CV, train)",
    template="plotly_white",
    height=500,
    width=550,
)

display(HTML(fig.to_html(include_plotlyjs="cdn")))

**`recall@topK`** — métrique orientée décision : si l'équipe recouvrement ne peut relancer qu'une partie du portefeuille (contrainte de capacité), quelle proportion des vrais défauts est capturée dans les K% de clients au score le plus élevé ? Contrairement au Recall classique (basé sur un seuil de probabilité), celle-ci est basée sur un volume de clients — directement actionnable pour la stratégie de relance. Fonction réutilisable définie dans `src/metrics.py`.

```python
def recall_at_k(y_true, y_proba, k):
    """Recall parmi les k% de clients au score de probabilite le plus eleve."""
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n_top = int(np.ceil(len(y_proba) * k))
    top_indices = np.argsort(y_proba)[::-1][:n_top]
    return y_true[top_indices].sum() / y_true.sum()
```

In [8]:
from src.metrics import recall_at_k

for k in [0.1, 0.2, 0.3]:
    n_clients = int(len(y_train) * k)
    print(f"recall@{int(k*100)}% : {recall_at_k(y_train, y_proba_cv, k):.3f}  ({n_clients} clients relancés)")

recall@10% : 0.321  (237 clients relancés)
recall@20% : 0.530  (474 clients relancés)
recall@30% : 0.648  (711 clients relancés)


##### **Interprétation des résultats**

- **Recall (0,624)** : le modèle détecte environ **62 %** des vrais défauts au seuil par défaut de 0,5 (317 détectés sur 508, 191 manqués) — c'est la métrique la plus importante ici, et `class_weight="balanced"` la pousse clairement à l'avantage du recall par rapport à la precision.
- **Precision (0,486)** : parmi les clients signalés à risque, ~49 % font effectivement défaut (335 fausses alertes, 317 vrais positifs).
- **F1-score (0,547)** et **Accuracy (0,778)** : compromis raisonnable pour un modèle linéaire simple, sans hyperparamètres optimisés.
- **PR-AUC (0,550)** vs **0,214** pour un modèle aléatoire (le taux de défaut) : plus de deux fois meilleur que l'aléatoire.
- **Matrice de confusion** : 317 vrais défauts détectés, 191 manqués, 335 fausses alertes, 1529 vrais négatifs.
- **`recall@topK`** : avec seulement les **10 %** de clients les plus à risque (237 clients), on capture déjà **32 %** des vrais défauts ; à **20 %** (474 clients), **53 %** ; à **30 %** (711 clients), **65 %**. Cette lecture est directement actionnable pour la direction Recouvrement & Risque : elle dit concrètement "si on ne relance que X % du portefeuille, on rattrape Y % des défauts", sans dépendre d'un seuil de probabilité arbitraire.

Ces chiffres sont cohérents avec la ligne "Logistic Regression" de la comparaison de modèles en section 5 (même modèle, même méthodologie de validation croisée) — un signe de cohérence interne du notebook plutôt qu'une coïncidence.

**Limite principale de ce baseline** : reste un modèle linéaire simple, sans hyperparamètres optimisés — ces chiffres viennent d'une seule répartition de validation croisée (pas de CV répétée), à interpréter avec prudence au 3e chiffre décimal près. Le choix définitif du seuil de décision (probabilité ou top K%) est volontairement **reporté à la fin de la modélisation**, une fois le modèle avancé du notebook 03 disponible, pour l'aligner sur le meilleur modèle et sur un vrai coût métier plutôt que sur ce baseline. Ces scores servent de **point de comparaison chiffré** pour la suite, sans jamais avoir touché `X_test`.

### 5. Présélection de modèles candidats

On teste maintenant 6 familles de modèles différentes pour voir si l'une d'elles capture mieux le signal que la régression logistique seule. Deux règles pour rester méthodologiquement rigoureux :

- **Évaluation uniquement sur `X_train`/`y_train`**, via validation croisée stratifiée (5 folds) — `X_test` n'est touché à aucun moment de cette étape, pour ne pas le "brûler" en le regardant à répétition pendant la sélection.
- **Pas d'optimisation d'hyperparamètres à ce stade** : chaque modèle est testé avec des réglages par défaut raisonnables (`class_weight="balanced"` quand disponible). Le but ici est de présélectionner des familles de modèles prometteuses, pas de trouver les meilleurs réglages — ça viendra dans le notebook 03, uniquement sur les finalistes retenus ici.

In [9]:
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from lightgbm import LGBMClassifier

SEED = 42

clf_candidats = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "SVM (noyau RBF)": svm.SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED),
    "Arbre de décision": DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "LightGBM": LGBMClassifier(class_weight="balanced", random_state=SEED, verbose=-1),
}

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = []
for name, model in clf_candidats.items():
    pipeline = build_pipeline(model)
    y_proba_cv = cross_val_predict(pipeline, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
    y_pred_cv = (y_proba_cv >= 0.5).astype(int)
    results.append({
        "Modèle": name,
        "F1": f1_score(y_train, y_pred_cv),
        "Recall": recall_score(y_train, y_pred_cv),
        "Precision": precision_score(y_train, y_pred_cv),
        "PR-AUC": average_precision_score(y_train, y_proba_cv),
        "recall@20%": recall_at_k(y_train, y_proba_cv, 0.2),
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False).reset_index(drop=True)
results_df.round(3)

c:\Users\abdou\OneDrive\Documents\Mes projets\Talk_to_my_data\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\abdou\OneDrive\Documents\Mes projets\Talk_to_my_data\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\abdou\OneDrive\Documents\Mes projets\Talk_to_my_data\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\abdou\OneDrive\Documents\Mes projets\Tal

,Modèle,F1,Recall,Precision,PR-AUC,recall@20%
0,Random Forest,0.550,0.528,0.575,0.562,0.528
1,Logistic Regression,0.547,0.624,0.486,0.550,0.530
2,LightGBM,0.526,0.506,0.548,0.563,0.510
3,Gradient Boosting,0.522,0.417,0.697,0.586,0.537
4,SVM (noyau RBF),0.515,0.439,0.623,0.533,0.531
5,Arbre de décision,0.424,0.425,0.423,0.303,0.400


##### **Résultats de la validation croisée (5 folds, train uniquement)**

Scores calculés sur les prédictions hors-échantillon agrégées (`cross_val_predict`), pas sur une moyenne de scores par fold — chaque client du train est prédit une seule fois, par le modèle entraîné sans lui, puis les métriques sont calculées sur l'ensemble agrégé.

| Modèle | F1 | Recall | Precision | PR-AUC | recall@20% |
|---|---|---|---|---|---|
| Random Forest | 0,550 | 0,528 | 0,575 | 0,562 | 0,528 |
| Logistic Regression | 0,547 | 0,624 | 0,486 | 0,550 | 0,530 |
| LightGBM | 0,526 | 0,506 | 0,548 | 0,563 | 0,510 |
| Gradient Boosting | 0,522 | 0,417 | **0,697** | **0,586** | **0,537** |
| SVM (noyau RBF) | 0,515 | 0,439 | 0,623 | 0,533 | 0,531 |
| Arbre de décision | 0,424 | 0,425 | 0,423 | 0,303 | 0,400 |

**Constats** :
- **PR-AUC est la métrique prioritaire** (cahier des charges) : classée strictement selon ce critère, Gradient Boosting domine (0,586), suivi de LightGBM (0,563) et Random Forest (0,562, quasi ex æquo — écart de 0,001, non significatif sur un seul run de CV), puis Logistic Regression (0,550).
- Gradient Boosting a aussi le meilleur `recall@20%` (0,537), malgré son Recall le plus faible au seuil 0,5 (0,417) — son classement des clients par probabilité est très pertinent, même si son seuil par défaut est trop conservateur.
- Random Forest domine nettement LightGBM sur toutes les métriques secondaires (F1 0,550 vs 0,526, Precision 0,575 vs 0,548, recall@20% 0,528 vs 0,510), malgré un PR-AUC quasi identique.
- L'arbre de décision seul reste nettement en retrait sur toutes les métriques.

**Sélection élargie à 4 finalistes** : plutôt que de trancher arbitrairement entre Logistic Regression et LightGBM (proches sur des métriques différentes), on garde les deux et on élargit la short-list :
- **Gradient Boosting** : meilleur PR-AUC (prioritaire) et meilleur `recall@20%`.
- **Random Forest** : PR-AUC quasi équivalent à LightGBM, mais nettement meilleur sur F1/Precision/recall@20%.
- **LightGBM** : 2e meilleur PR-AUC, à égalité quasi parfaite avec Random Forest sur ce critère prioritaire.
- **Logistic Regression** : en retrait sur PR-AUC (0,550) mais meilleur Recall du haut de classement (0,624) et seule option linéaire/interprétable — gardée pour la diversité et comme référence simple face aux 3 modèles à base d'arbres.

**Écarté** : SVM RBF (dominé sur PR-AUC et la plupart des autres métriques par les 4 finalistes retenus).

### 6. Synthèse

**Ce qui a été fait dans ce notebook** :
- Centralisation de la préparation des données (`src/config.py`, `src/data_prep.py`) : chargement, correction du sens chronologique (`pay_0`→`pay_1`), nettoyage des modalités, feature engineering (`reste_du_moyen`, `taux_utilisation_moyen`, `bill_amt_moyen`/`tendance`, `retard_max`/`nb_mois_en_retard`), encodage ordinal d'`education_level`.
- Un modèle baseline (`LogisticRegression`, `class_weight="balanced"`) entraîné et évalué avec des métriques adaptées au déséquilibre (Accuracy, Precision, Recall, F1, PR-AUC, matrice de confusion, `recall@topK`), par validation croisée sur le train (comparée à la résubstitution pour un premier repère de sur-apprentissage), sans jamais toucher `X_test`.
- Une présélection de 6 familles de modèles par validation croisée sur le train uniquement (`X_test` jamais touché), aboutissant à 4 finalistes : **Gradient Boosting, Random Forest, LightGBM, Logistic Regression**.
- Une justification explicite du choix `class_weight` plutôt que du ré-échantillonnage.

**Limites de ce qui a été fait ici** :
- Les modèles présélectionnés utilisent des hyperparamètres par défaut, non optimisés — le classement actuel (notamment l'écart LightGBM/Random Forest sur PR-AUC, non significatif) pourrait changer après tuning.
- Une seule répartition de validation croisée (pas de CV répétée) : les scores ont une variance d'échantillonnage non quantifiée, à interpréter avec prudence au 3e chiffre décimal près.
- Le seuil de décision (probabilité ou top K%) n'a volontairement pas été fixé — resté au défaut de 0,5 pour les métriques ponctuelles, avec `recall@topK` calculé à titre indicatif sur plusieurs K.
- Le modèle baseline (régression logistique) reste un point de comparaison, pas la solution retenue.

**Pistes pour le notebook 03 (modélisation avancée)** :
1. Optimisation des hyperparamètres des 4 finalistes (`GridSearchCV`/`RandomizedSearchCV`) sur le train, toujours sans toucher au test.
2. Sélection du modèle final selon PR-AUC (métrique prioritaire) et `recall@topK`.
3. Choix définitif du seuil de décision (stratégie top K% ou seuil de probabilité), aligné sur un coût métier explicite.
4. Une seule évaluation finale sur `X_test`/`y_test`, sur le modèle unique retenu.
5. Sauvegarde du modèle et production du fichier de scoring (`id`, `proba_default`, `label_pred`) sur le jeu de test — dans `src/train.py`/`src/infer.py`.